In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# EDA + Encoding + Scaling + Train test split + basic model

## EDA

In [ ]:
df = sns.load_dataset("titanic")
df

In [ ]:
dfNullColumns = df.isnull().sum().to_frame("Null Percentage")
dfNullColumns["Null Percentage"] = dfNullColumns["Null Percentage"] / df.shape[0] * 100
dfNullColumns

In [ ]:
columnsWithNullValues = dfNullColumns[dfNullColumns["Null Percentage"] > 0]
columnsWithNullValues

In [ ]:
columnsToBeDeleted = dfNullColumns[dfNullColumns["Null Percentage"] > 20]
columnsToBeDeleted

In [ ]:
dropColumns = columnsToBeDeleted.index
dropColumns

In [ ]:
dfAfterEDA = df.drop(columns = dropColumns)
dfAfterEDA

In [ ]:
dfAfterEDA["who"].unique()

In [ ]:
dropRepeatedColumns = ["embarked", "class", "adult_male", "alive", "sibsp", "parch"]
dfAfterEDA = dfAfterEDA.drop(columns = dropRepeatedColumns)
dfAfterEDA

In [ ]:
dfAfterEDA["embark_town"].unique()

In [ ]:
dfAfterEDA["embark_town"].isnull().sum()

In [ ]:
dfAfterEDA = dfAfterEDA.dropna(subset = ["embark_town"])
dfAfterEDA["embark_town"].isnull().sum()

In [ ]:
def sum_nan_age_by_who(df):
    result = df.groupby('who')['age'].apply(lambda x: x.isna().sum())
    print(result)
sum_nan_age_by_who(dfAfterEDA)

In [ ]:
dfAfterEDA.loc[:, "age"] = dfAfterEDA.groupby('who')['age'].transform(lambda x: x.fillna(x.median()))
dfAfterEDA

## Encoding: -
- Encoding is the process of converting categorical (text) data into numerical form so that machine learning algorithms can understand and process it. Most ML models (especially in libraries like Scikit-learn) work only with numbers, not strings.
### ColumnTransformer
- Used to apply encoding to specific columns
### Label Encoding
- Converts each category into a unique integer.
- Example:
"Low", "Medium", "High" → 0, 1, 2
- Suitable for target variables (y)
- Not ideal for input features (X) because it introduces false order
### Ordinal Encoding
- Assigns integers to categories with a meaningful order
- Example:
"Low" < "Medium" < "High" → 0, 1, 2
- Use when categories have natural ranking
- Works on multiple columns at once
### One-Hot Encoding
- Creates binary columns (0/1) for each category.
- Example:
Color → Red, Blue, Green
→ [1,0,0], [0,1,0], [0,0,1]
- drop='first' → avoids dummy variable trap
- sparse=False → returns dense array
- Best for nominal data (no order)
- Most commonly used encoding
### Binary Encoding (Not directly in sklearn)
- Not native in sklearn, but used via external libraries like category_encoders.
- Converts categories → integers → binary representation
- When categories are high in number (reduces dimensionality vs one-hot)
### Target Encoding (Mean Encoding)
- Replaces categories with mean of target variable
- Example:
City → average house price
- Risk of data leakage
### Frequency / Count Encoding
- Replaces category with its frequency/count
- Example:
"Delhi" appears 100 times → 100
- Done using pandas (value_counts())

In [ ]:
dfAfterEDA

In [ ]:
dfAfterEDA["alone"] = dfAfterEDA["alone"].astype(int)
dfAfterEDA

Note: - OneHotEncoding can be done with 2 methods, 
1. drop = "First" - creates categories of 0/1 and removes all other classes. eg - male = 0, female = 1, trans - deleted. Best for Regression.
2. drop = none - does not remove any class, rather creates new columns. eg - male = [1, 0, 0], female = [0, 1, 0], trans = [0, 0, 1]. Best for Randomforest

In [ ]:
OneHot1 = OneHotEncoder(sparse_output = False, drop = "first")

In [ ]:
sexOneHotEncoded = OneHot1.fit_transform(dfAfterEDA[["sex"]])
sexOneHotEncodedDf = pd.DataFrame(sexOneHotEncoded, index = dfAfterEDA.index)
dfAfterEDA["sex"] = sexOneHotEncodedDf
dfAfterEDA

In [ ]:
OneHot2 = OneHotEncoder(drop = None, sparse_output = False)

In [ ]:
whoOneHotEncoded = OneHot2.fit_transform(dfAfterEDA[["who"]])
whoOneHotEncodedDf = pd.DataFrame(whoOneHotEncoded, columns = OneHot2.get_feature_names_out(["who"]), index = dfAfterEDA.index)
dfAfterEDA = pd.concat([dfAfterEDA, whoOneHotEncodedDf], axis = 1)
dfAfterEDA = dfAfterEDA.drop(columns = ["who"])
dfAfterEDA

In [ ]:
embark_townOneHotEncoded = OneHot2.fit_transform(dfAfterEDA[["embark_town"]])
embark_townOneHotEncodedDf = pd.DataFrame(embark_townOneHotEncoded, columns = OneHot2.get_feature_names_out(["embark_town"]), index = dfAfterEDA.index)
dfAfterEDA = pd.concat([dfAfterEDA, embark_townOneHotEncodedDf], axis = 1)
dfAfterEDA = dfAfterEDA.drop(columns = ["embark_town"])
dfAfterEDA

In [ ]:
dfAfterEDA.dtypes

## Scaling:- 
- Feature scaling is the process of transforming numerical features so that they lie within a similar range or scale. This ensures that no single feature dominates others due to its magnitude.
### Why Scaling is Important?
- Prevents features with large values from dominating the model
- Improves convergence speed of algorithms
- Essential for distance-based models (KNN, SVM, K-Means)
- Helps gradient-based models perform better
### When Scaling is NOT Required
- Tree-based models (e.g., Random Forest, Decision Trees)
- Models that are not distance-based
### Standardization (Z-score Normalization)
- Transforms data to have:
Mean = 0
Standard deviation = 1
- Centers data around zero
- Handles outliers better than normalization
### Min-Max Scaling (Normalization)
- Scales data to a fixed range (usually 0 to 1)
- Preserves distribution shape
- Sensitive to outliers
### Robust Scaling
- Uses median and interquartile range (IQR)
- Resistant to outliers
- Works well for skewed data
### Max Absolute Scaling
- Scales data to range [-1, 1]
- Preserves sparsity
- Useful for sparse datasets
### Unit Vector Scaling (Normalization by Norm)
- Scales each sample to have unit norm
- Used in text classification
- Important for cosine similarity
### Best Practices
- Always fit scaler on training data only
- Apply same scaler to test data
- Use StandardScaler as default
- Use RobustScaler if outliers exist
- Skip scaling for tree-based models

## To find Outliers Which Method Should You Use?
- Boxplot - Quick visualization
- IQR - Most reliable (recommended)
- Z-score - Normally distributed data

In [ ]:
sns.boxplot(x=dfAfterEDA['fare'])
plt.show()
sns.boxplot(x=dfAfterEDA['age'])
plt.show()

In [ ]:
RobustScaler = RobustScaler()

In [ ]:
dfAfterEDA["fare"] = RobustScaler.fit_transform(dfAfterEDA[["fare"]])

In [ ]:
sns.boxplot(x=dfAfterEDA['fare'])
plt.show()

In [ ]:
x = dfAfterEDA.drop(columns = ["survived"])
y = dfAfterEDA["survived"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = .2, random_state = 12)

In [ ]:
model1 = RandomForestClassifier(random_state=40)
model1.fit(x_train, y_train)

In [ ]:
model2 = LinearRegression()
model2.fit(x_train, y_train)